# 🦅 Bird Classifier - Model Summary

Load and display the model architecture for Stage 1 and Stage 2 training.

In [33]:
import subprocess
import sys

print("📦 Installing required packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchsummary"])
print("✅ Packages installed successfully!")

📦 Installing required packages...
✅ Packages installed successfully!


In [ ]:
import torch
from pathlib import Path
import sys

# Add project to path
sys.path.insert(0, str(Path.cwd()))

from config import NUM_CLASSES, CHECKPOINT_DIR, DEVICE

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [67]:
print("\n" + "="*110)
print("STAGE 1:MODEL ARCHITECTURE TABLE")
print("="*110)

# Get actual layers from the model
print(f"\n{'Layer (Type)':<35} {'Output Shape':<20} {'Param #':<15}")
print("-" * 110)

# EfficientNet-B4 backbone
backbone_params = sum(p.numel() for p in model_stage1.backbone.parameters())
backbone_trainable = sum(p.numel() for p in model_stage1.backbone.parameters() if p.requires_grad)
print(f"{'EfficientNet-B4 (Backbone)':<35} {'(B, 1792, 8, 8)':<20} {backbone_params:,}{'':>3} ")

# Custom head layers
print(f"{'Flatten':<35} {'(B, 1792)':<20} {'0'} ")
print(f"{'Dropout (p=0.5)':<35} {'(B, 1792)':<20} {'0'} ")

# Get actual layer parameters from classifier
for name, param in model_stage1.classifier.named_parameters():
    layer_idx = int(name.split('.')[0])
    layer = model_stage1.classifier[layer_idx]
    layer_name = str(layer).split('(')[0].strip()
    param_count = param.numel()
    
    if "Linear" in layer_name:
        if "2" in name.split('.')[0]:
            print(f"{'Linear (1792 → 512)':<35} {'(B, 512)':<20} {param_count:,}")
        elif "6" in name.split('.')[0]:
            print(f"{'Linear (512 → 256)':<35} {'(B, 256)':<20} {param_count:,}")
        elif "10" in name.split('.')[0]:
            print(f"{'Linear (256 → 25)':<35} {'(B, 25)':<20} {param_count:,}")
    elif "BatchNorm" in layer_name:
        if "3" in name.split('.')[0]:
            print(f"{'BatchNorm1d (512)':<35} {'(B, 512)':<20} {param_count:,}")
        elif "7" in name.split('.')[0]:
            print(f"{'BatchNorm1d (256)':<35} {'(B, 256)':<20} {param_count:,}")

# Add activation and dropout layers (0 params)
print(f"{'GELU':<35} {'(B, 512)':<20} {'0'}")
print(f"{'Dropout (p=0.5)':<35} {'(B, 512)':<20} {'0'}")
print(f"{'GELU':<35} {'(B, 256)':<20} {'0'}")
print(f"{'Dropout (p=0.25)':<35} {'(B, 256)':<20} {'0'}")

print("-" * 110)

print("\n" + "="*110)
print("PARAMETER SUMMARY:")
print("="*110)
total_params = sum(p.numel() for p in model_stage1.parameters())
trainable_params = sum(p.numel() for p in model_stage1.parameters() if p.requires_grad)
non_trainable_params = total_params - trainable_params

print(f"Trainable params (Custom Head):     {trainable_params:,}")
print(f"Non-trainable params (Backbone):   {non_trainable_params:,}")
print(f"Total params:                      {total_params:,}")
print("="*110)


STAGE 1:MODEL ARCHITECTURE TABLE

Layer (Type)                        Output Shape         Param #        
--------------------------------------------------------------------------------------------------------------
EfficientNet-B4 (Backbone)          (B, 1792, 8, 8)      17,548,616    
Flatten                             (B, 1792)            0 
Dropout (p=0.5)                     (B, 1792)            0 
Linear (1792 → 512)                 (B, 512)             917,504
Linear (1792 → 512)                 (B, 512)             512
BatchNorm1d (512)                   (B, 512)             512
BatchNorm1d (512)                   (B, 512)             512
Linear (512 → 256)                  (B, 256)             131,072
Linear (512 → 256)                  (B, 256)             256
BatchNorm1d (256)                   (B, 256)             256
BatchNorm1d (256)                   (B, 256)             256
Linear (256 → 25)                   (B, 25)              6,400
Linear (256 → 25)             

In [68]:
print("\n" + "="*110)
print("STAGE 2:MODEL ARCHITECTURE TABLE")
print("="*110)

# Get actual layers from the model
print(f"\n{'Layer (Type)':<35} {'Output Shape':<20} {'Param #':<15} ")
print("-" * 110)

# EfficientNet-B4 backbone (TRAINABLE in Stage 2)
backbone_params = sum(p.numel() for p in model_stage2.backbone.parameters())
backbone_trainable = sum(p.numel() for p in model_stage2.backbone.parameters() if p.requires_grad)
print(f"{'EfficientNet-B4 (Backbone)':<35} {'(B, 1792, 8, 8)':<20} {backbone_params:,}{'':>3} ")

# Custom head layers
print(f"{'Flatten':<35} {'(B, 1792)':<20} {'0'} ")
print(f"{'Dropout (p=0.5)':<35} {'(B, 1792)':<20} {'0'} ")

# Get actual layer parameters from classifier
for name, param in model_stage2.classifier.named_parameters():
    layer_idx = int(name.split('.')[0])
    layer = model_stage2.classifier[layer_idx]
    layer_name = str(layer).split('(')[0].strip()
    param_count = param.numel()
    
    if "Linear" in layer_name:
        if "2" in name.split('.')[0]:
            print(f"{'Linear (1792 → 512)':<35} {'(B, 512)':<20} {param_count:,}")
        elif "6" in name.split('.')[0]:
            print(f"{'Linear (512 → 256)':<35} {'(B, 256)':<20} {param_count:,}")
        elif "10" in name.split('.')[0]:
            print(f"{'Linear (256 → 25)':<35} {'(B, 25)':<20} {param_count:,}")
    elif "BatchNorm" in layer_name:
        if "3" in name.split('.')[0]:
            print(f"{'BatchNorm1d (512)':<35} {'(B, 512)':<20} {param_count:,}")
        elif "7" in name.split('.')[0]:
            print(f"{'BatchNorm1d (256)':<35} {'(B, 256)':<20} {param_count:,}")

# Add activation and dropout layers (0 params)
print(f"{'GELU':<35} {'(B, 512)':<20} {'0'}")
print(f"{'Dropout (p=0.5)':<35} {'(B, 512)':<20} {'0'}")
print(f"{'GELU':<35} {'(B, 256)':<20} {'0'}")
print(f"{'Dropout (p=0.25)':<35} {'(B, 256)':<20} {'0'}")

print("-" * 110)

print("\n" + "="*110)
print("PARAMETER SUMMARY:")
print("="*110)
total_params = sum(p.numel() for p in model_stage2.parameters())
trainable_params = sum(p.numel() for p in model_stage2.parameters() if p.requires_grad)
non_trainable_params = total_params - trainable_params

print(f"Trainable params (All Layers):      {trainable_params:,}")
print(f"Non-trainable params:              {non_trainable_params:,}")
print(f"Total params:                      {total_params:,}")
print("="*110)


STAGE 2:MODEL ARCHITECTURE TABLE

Layer (Type)                        Output Shape         Param #         
--------------------------------------------------------------------------------------------------------------
EfficientNet-B4 (Backbone)          (B, 1792, 8, 8)      17,548,616    
Flatten                             (B, 1792)            0 
Dropout (p=0.5)                     (B, 1792)            0 
Linear (1792 → 512)                 (B, 512)             917,504
Linear (1792 → 512)                 (B, 512)             512
BatchNorm1d (512)                   (B, 512)             512
BatchNorm1d (512)                   (B, 512)             512
Linear (512 → 256)                  (B, 256)             131,072
Linear (512 → 256)                  (B, 256)             256
BatchNorm1d (256)                   (B, 256)             256
BatchNorm1d (256)                   (B, 256)             256
Linear (256 → 25)                   (B, 25)              6,400
Linear (256 → 25)            

## Training Code - Replicating Your Laptop Training

This section shows how to replicate the training process from scratch. You can run this on Google Colab or your local machine.


In [ ]:
import torch.nn as nn
import torch.optim as optim
import math
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
import time

# ============================================================================
# DATA SETUP
# ============================================================================

# Image preprocessing (matching actual training config)
IMAGE_SIZE = 224  # From config.py

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load dataset (adjust path to your data directory)
# data_dir should contain folders: bird_species_1/, bird_species_2/, etc.
print("✅ Data transforms configured:")
print(f"   - Train: RandomResizedCrop, flip, rotation, color jitter, blur, grayscale")
print(f"   - Val: Resize only")
print(f"   - Input size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"   - Normalize: ImageNet mean/std")
print(f"   - Batch size: 32")
print(f"\nNote: Create train_loader and val_loader from your dataset before running training")


In [ ]:
# ============================================================================
# MODEL ARCHITECTURE DEFINITION 
# ============================================================================

class BirdClassifier(nn.Module):
    """EfficientNet-B4 + custom classification head for 25 bird species"""
    
    def __init__(self, num_classes: int = 25, dropout_rate: float = 0.4):
        super(BirdClassifier, self).__init__()
        
        from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
        
        # Backbone: ImageNet pretrained
        weights = EfficientNet_B4_Weights.IMAGENET1K_V1
        self.backbone = efficientnet_b4(weights=weights)
        self.backbone.classifier = nn.Identity()  # Remove original head
        
        # Custom classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(1792, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(p=dropout_rate * 0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(self.backbone(x))
    
    def freeze_backbone(self, freeze=True):
        for param in self.backbone.parameters():
            param.requires_grad = not freeze


print("✅ Model architecture defined:")
print("   Backbone: EfficientNet-B4 (ImageNet pretrained)")
print("   Custom Head: 1792 → 512 → 256 → 25")
print("\n   Stage 1: Backbone FROZEN, Head TRAINED (LR=1e-3, 8 epochs)")
print("   Stage 2: Backbone UNFROZEN, Full model FINE-TUNED (LR=1e-4, 15 epochs)")
print("   Loss: CrossEntropyLoss (label_smoothing=0.1)")
print("   Optimizer: AdamW (weight_decay=1e-4)")


In [ ]:
# ============================================================================
# TRAINING UTILITIES
# ============================================================================

def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

def validate(model, val_loader, criterion, device):
    """Validate the model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_loss = running_loss / len(val_loader)
    val_acc = 100 * correct / total
    return val_loss, val_acc

print("✅ Training utilities defined (train_epoch, validate)")

In [ ]:
# ============================================================================
# STAGE 1: FEATURE EXTRACTION (FROZEN BACKBONE)
# ============================================================================
# Stage 1 trains only the custom classification head while keeping
# the EfficientNet-B4 backbone frozen (using its ImageNet features)

print("\n" + "="*80)
print("STAGE 1 TRAINING SETUP")
print("="*80)

# Create fresh model for training
train_model_stage1 = BirdClassifier(num_classes=NUM_CLASSES, dropout_rate=0.4).to(device)

# Freeze backbone (feature extraction only) - backbone parameters won't be updated
train_model_stage1.freeze_backbone(freeze=True)

# Loss function with label smoothing (from config: LABEL_SMOOTHING = 0.1)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Stage 1 Optimizer: Train only classifier head parameters
# Using AdamW as in the actual training (weight_decay=1e-4)
optimizer_stage1 = optim.AdamW(
    train_model_stage1.classifier.parameters(),  # Only head is trainable
    lr=1e-3,  # Stage 1 LR (from config: STAGE1_LR = 1e-3)
    weight_decay=1e-4
)

# Learning rate scheduler: Warmup (3 epochs) + Cosine annealing
def get_warmup_cosine_scheduler(optimizer, total_epochs, warmup_epochs=3):
    """Create warmup + cosine annealing scheduler (matches actual training)"""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch) / float(max(1, warmup_epochs))
        else:
            progress = float(epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
            return max(1e-5, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scheduler_stage1 = get_warmup_cosine_scheduler(optimizer_stage1, total_epochs=8)

# Training configuration (from config file)
STAGE1_EPOCHS = 8
BATCH_SIZE = 32

print(f"\nStage 1 Configuration:")
print(f"  Epochs: {STAGE1_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: 1e-3")
print(f"  Optimizer: AdamW (weight_decay=1e-4)")
print(f"  Loss: CrossEntropyLoss (label_smoothing=0.1)")
print(f"  Scheduler: Warmup (3 epochs) + Cosine annealing")
print(f"  Backbone: FROZEN (EfficientNet-B4)")
print(f"  Training: Classifier head only")
print(f"  Trainable parameters: {sum(p.numel() for p in train_model_stage1.parameters() if p.requires_grad):,}")
print(f"  Frozen parameters: {sum(p.numel() for p in train_model_stage1.parameters() if not p.requires_grad):,}")

# Stage 1 Training Loop
import math
stage1_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc_stage1 = 0

for epoch in range(STAGE1_EPOCHS):
    train_loss, train_acc = train_epoch(train_model_stage1, train_loader, criterion, optimizer_stage1, device)
    val_loss, val_acc = validate(train_model_stage1, val_loader, criterion, device)
    
    stage1_history['train_loss'].append(train_loss)
    stage1_history['train_acc'].append(train_acc)
    stage1_history['val_loss'].append(val_loss)
    stage1_history['val_acc'].append(val_acc)
    
    if val_acc > best_val_acc_stage1:
        best_val_acc_stage1 = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': train_model_stage1.state_dict(),
            'optimizer_state_dict': optimizer_stage1.state_dict(),
            'best_val_acc': best_val_acc_stage1
        }, 'best_model_stage1.pt')
    
    scheduler_stage1.step()
    current_lr = optimizer_stage1.param_groups[0]['lr']
    
    print(f'Epoch [{epoch+1}/{STAGE1_EPOCHS}] '
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}% | '
          f'LR: {current_lr:.2e}')

print("\n✅ Stage 1 Training Complete")


In [ ]:
# ============================================================================
# STAGE 2: FINE-TUNING (UNFROZEN BACKBONE)
# ============================================================================
# Stage 2 unfreezes the backbone and fine-tunes the entire model
# with a lower learning rate for careful adjustment of all parameters

print("\n" + "="*80)
print("STAGE 2 TRAINING SETUP")
print("="*80)

# Create fresh model starting from Stage 1 checkpoint
train_model_stage2 = BirdClassifier(num_classes=NUM_CLASSES, dropout_rate=0.4).to(device)

# Try to load Stage 1 checkpoint as baseline
try:
    checkpoint = torch.load(Path(CHECKPOINT_DIR) / "best_model_stage1.pt", map_location=device, weights_only=False)
    train_model_stage2.load_state_dict(checkpoint['model_state_dict'])
    print("✓ Loaded Stage 1 checkpoint as baseline")
except:
    print("⚠ Stage 1 checkpoint not found. Training from scratch.")

# UNFREEZE ALL LAYERS for fine-tuning
train_model_stage2.freeze_backbone(freeze=False)

# Loss function (same as Stage 1)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Stage 2 Optimizer: Fine-tune entire model with lower learning rate
# Using AdamW (same as Stage 1) with lower LR for stability
optimizer_stage2 = optim.AdamW(
    train_model_stage2.parameters(),  # All parameters trainable
    lr=1e-4,  # Stage 2 LR (from config: STAGE2_LR = 1e-4)
    weight_decay=1e-4
)

# Same scheduler as Stage 1 (Warmup + Cosine annealing)
STAGE2_EPOCHS = 15  # From config: STAGE2_EPOCHS = 15
scheduler_stage2 = get_warmup_cosine_scheduler(optimizer_stage2, total_epochs=STAGE2_EPOCHS)

print(f"\nStage 2 Configuration:")
print(f"  Epochs: {STAGE2_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: 1e-4 (lower than Stage 1 for stability)")
print(f"  Optimizer: AdamW (weight_decay=1e-4)")
print(f"  Loss: CrossEntropyLoss (label_smoothing=0.1)")
print(f"  Scheduler: Warmup (3 epochs) + Cosine annealing")
print(f"  Backbone: UNFROZEN (fine-tuning)")
print(f"  Training: Full model")
print(f"  Trainable parameters: {sum(p.numel() for p in train_model_stage2.parameters() if p.requires_grad):,}")
print(f"  Total parameters: {sum(p.numel() for p in train_model_stage2.parameters()):,}")

# Stage 2 Training Loop
stage2_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc_stage2 = 0

for epoch in range(STAGE2_EPOCHS):
    train_loss, train_acc = train_epoch(train_model_stage2, train_loader, criterion, optimizer_stage2, device)
    val_loss, val_acc = validate(train_model_stage2, val_loader, criterion, device)
    
    stage2_history['train_loss'].append(train_loss)
    stage2_history['train_acc'].append(train_acc)
    stage2_history['val_loss'].append(val_loss)
    stage2_history['val_acc'].append(val_acc)
    
    if val_acc > best_val_acc_stage2:
        best_val_acc_stage2 = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': train_model_stage2.state_dict(),
            'optimizer_state_dict': optimizer_stage2.state_dict(),
            'best_val_acc': best_val_acc_stage2
        }, 'best_model_stage2.pt')
    
    scheduler_stage2.step()
    current_lr = optimizer_stage2.param_groups[0]['lr']
    
    print(f'Epoch [{epoch+1}/{STAGE2_EPOCHS}] '
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}% | '
          f'LR: {current_lr:.2e}')

print("\n✅ Stage 2 Training Complete")
